# Odd Lots long-only baskets

This notebook consumes the public, judge-approved call list. Admission is a bullish US-listed company call with an identified security and an entry price; there are no additional confidence, model-count, or quote-score cuts. Unresolved calls remain in the ledger and make the coverage status incomplete.

Buy at the close of the first trading session strictly after publication (t+1). Rebalance when a new or renewed call enters or an active call expires. Each active stock receives min(1 / active names, 10%) of portfolio value; the remainder stays in Treasury cash. The 10% limit applies at allocation only: weights drift between events. There is no daily or monthly rebalance. Three- and twelve-month calls exit at the close of the first trading session on or after their expiry; renewals reset the clock. Existing positions earn returns through the execution close; new positions earn subsequent returns only.

Cash earns the prior session's 3M Treasury yield over elapsed calendar days. Everything is a retrospective backcast. The arithmetic lives in `scripts/backtest_utils.py` and has focused regression tests. Cached Yahoo prices and all locally generated market results belong in the ignored private directory. No market-result CSVs are included in the public repository. The notebook works offline from those caches. Set `ODD_LOTS_ALLOW_NETWORK=1` explicitly to fetch missing inputs. See `notes/BASKET_PLAN.md` for limitations.


In [ ]:
import hashlib
import json
import os
import platform
from importlib.metadata import version
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts" / "backtest_utils.py").exists())
PRIVATE = Path(os.getenv("ODD_LOTS_PRIVATE_DIR") or ROOT / "private").expanduser()
if not PRIVATE.is_absolute():
    PRIVATE = ROOT / PRIVATE
ALLOW_NETWORK = os.getenv("ODD_LOTS_ALLOW_NETWORK", "0") == "1"
REFRESH = False  # With network enabled, refresh all requested prices and Treasury observations.
sys.path.insert(0, str(ROOT / "scripts"))
from backtest_utils import BENCHMARKS, clean_ticker, read_price_inputs, prepare_calls, export_results


## Inputs and explicit coverage

The public input includes unresolved identities such as ATVI. A current security directory does not establish historical membership or acquisition proceeds. Missing history is reported, never treated as proof of delisting. A missing price after a position is held stops calculation rather than fabricating cash proceeds.


In [ ]:
raw_calls = pd.read_csv(ROOT / "data" / "trade_recommendations.csv")
symbols = pd.concat([raw_calls.ticker, raw_calls.get("proposed_ticker", pd.Series(dtype=str))])
normalized = [clean_ticker(t) for t in symbols]
tickers = sorted({t for t, kind in normalized if kind == "us"} | set(BENCHMARKS))
P, treasury_yields, missing_downloads, metadata = read_price_inputs(
    PRIVATE / "data", tickers, allow_network=ALLOW_NETWORK, refresh=REFRESH)
calls = prepare_calls(raw_calls, P, metadata)
print(f"{len(calls)} judge-approved ledger rows; {int(calls.admitted.sum())} admitted long calls")
print(f"Price observations: {P.index[0].date()} to {P.index[-1].date()}")
print("Missing download histories:", missing_downloads)
unresolved = calls.loc[calls.unresolved_eligible]
if len(unresolved):
    print(f"INCOMPLETE COVERAGE: {len(unresolved)} eligible calls need historical-security/price resolution.")
    display(unresolved[[c for c in ["publish_date", "company", "proposed_ticker", "ticker", "price_status"] if c in unresolved]])


## Baskets and leave-one-out sensitivity

A holds indefinitely, B exits after 3 months, and C after 12 months. Benchmarks and baskets all start at 100 at the first admitted call's t+1 close. Annual returns use elapsed calendar days. Leave-one-out removes all calls on each name but keeps the full basket's start and end dates; before any remaining name enters, it holds Treasury cash.

Returns are gross of trading costs and taxes. RSP is a comparison, not a membership constraint. The call universe includes MLPs and companies outside the S&P 500. Quote and security review is retrospective and not a point-in-time investment signal. Beta uses daily returns in excess of the modeled cash return against SPY's excess returns, with an intercept. Sharpe uses daily excess returns annualized with sqrt(252). The local chart displays A, B, C, RSP and Treasury cash; SPY remains an internal reference for the statistics.


In [ ]:
returns, summary, leave_one_out = export_results(calls, P, treasury_yields, PRIVATE / "baskets")
# Tie every export to exact inputs without distributing price or metadata caches.
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
metadata_path = PRIVATE / "baskets" / "run_metadata.json"
provenance = json.loads(metadata_path.read_text())
provenance["call_input_sha256"] = sha256(ROOT / "data" / "trade_recommendations.csv")
provenance["private_input_sha256"] = {name: sha256(PRIVATE / "data" / name)
    for name in ["yahoo_adj_close.csv", "DGS3MO.csv", "yahoo_meta.csv"]}
provenance["code_sha256"] = {name: sha256(ROOT / name)
    for name in ["scripts/backtest_utils.py", "notebooks/odd_lots_backtest.ipynb"]}
provenance["runtime"] = {"python": platform.python_version(), **{name: version(name) for name in ["pandas", "numpy", "yfinance"]}}
provenance["network_enabled"] = ALLOW_NETWORK
metadata_path.write_text(json.dumps(provenance, indent=2) + "\n")
# Generated results remain in PRIVATE / "baskets" for local analysis only.
display(summary)
display(leave_one_out.sort_values(["variant", "vs_spy"]).groupby("variant").head(3))


In [ ]:
import matplotlib.pyplot as plt

columns = ["A_ew_forever", "B_ew_3m_rolloff", "C_ew_12m_rolloff", "RSP", "BILL"]
ax = returns[columns].plot(figsize=(10, 5), title="Odd Lots baskets: retrospective returns, incomplete historical coverage")
ax.set_ylabel("Index level; common start = 100")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
